# Chains in LangChain

## Table of Contents
1. [What are Chains?](#what-are-chains)
2. [Why Use Chains?](#why-use-chains)
3. [Types of Chains](#types-of-chains)
4. [How to Use Chains](#how-to-use-chains)
5. [Real-World Examples](#real-world-examples)
6. [Best Practices](#best-practices)

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv(override=True)

llm = ChatAnthropic(model='claude-sonnet-4-6', max_tokens=400)

---
## 1. What are Chains?

A **Chain** is a sequence of components where the **output of one step becomes the input of the next**.

```
Input --> [Prompt] --> [LLM] --> [Parser] --> Output
```

In LangChain, chains are built using the **pipe operator `|`** (LCEL — LangChain Expression Language):

```python
chain = prompt | llm | output_parser
```

Each component is a **Runnable**, so the whole chain is also a Runnable —
meaning `.invoke()`, `.batch()`, `.stream()` all work on it.

---
## 2. Why Use Chains?

| Without Chains | With Chains |
|---|---|
| Manual step-by-step code | Declarative pipeline |
| Hard to reuse | Composable and reusable |
| Error-prone wiring | Clean, readable flow |
| Harder to debug | Easy to inspect each step |

> **Analogy:** Like an assembly line — each station does one job and passes the result to the next.

---
## 3. Types of Chains

| Type | Syntax | Use Case |
|---|---|---|
| **Simple Chain** | `prompt \| llm` | Basic prompt to response |
| **Chain with Parser** | `prompt \| llm \| parser` | Get clean string output |
| **Sequential Chain** | `chain1 \| transform \| chain2` | Multi-step pipeline |
| **Parallel Chain** | `RunnableParallel(a=c1, b=c2)` | Run multiple chains at once |


---
## 4. How to Use Chains

### 4a. Simple Chain — `prompt | llm`

In [ ]:
prompt = ChatPromptTemplate.from_template('Tell me a fun fact about {topic}.')

chain = prompt | llm          # prompt --> llm

response = chain.invoke({'topic': 'space'})
print(type(response).__name__)    # AIMessage
print(response.content)

### 4b. Chain with Output Parser — `prompt | llm | parser`

`StrOutputParser` extracts plain **string** from the AIMessage — no need to call `.content`.

In [ ]:
chain = prompt | llm | StrOutputParser()   # prompt --> llm --> parser

response = chain.invoke({'topic': 'black holes'})
print(type(response))    # <class 'str'>  (plain string, not AIMessage)
print(response)

### 4c. Sequential Chain — output of one chain feeds the next

In [ ]:
# Step 1: summarise a topic
summary_prompt = ChatPromptTemplate.from_template('Summarize {topic} in 2 sentences.')
summary_chain  = summary_prompt | llm | StrOutputParser()

# Step 2: turn that summary into a tweet
tweet_prompt   = ChatPromptTemplate.from_template('Turn this into a tweet:\n{summary}')
tweet_chain    = tweet_prompt | llm | StrOutputParser()

# Connect: summary output --> tweet input
full_chain = summary_chain | (lambda s: {'summary': s}) | tweet_chain

result = full_chain.invoke({'topic': 'Quantum Computing'})
print(result)

### 4d. Parallel Chain — run multiple chains simultaneously

In [ ]:
from langchain_core.runnables import RunnableParallel

pros_prompt = ChatPromptTemplate.from_template('List 2 pros of {technology}.')
cons_prompt = ChatPromptTemplate.from_template('List 2 cons of {technology}.')

# Both chains run IN PARALLEL
parallel_chain = RunnableParallel(
    pros = pros_prompt | llm | StrOutputParser(),
    cons = cons_prompt | llm | StrOutputParser(),
)

result = parallel_chain.invoke({'technology': 'AI'})
print('PROS:\n', result['pros'])
print()
print('CONS:\n', result['cons'])

---
## 5. Real-World Examples

### Example 1 — Translate then Summarize

In [ ]:
translate_prompt = ChatPromptTemplate.from_template('Translate to French: {text}')
summarize_prompt = ChatPromptTemplate.from_template('Summarize in one sentence: {translated}')

pipeline = (
    translate_prompt | llm | StrOutputParser()
    | (lambda t: {'translated': t})
    | summarize_prompt | llm | StrOutputParser()
)

result = pipeline.invoke({'text': 'LangChain helps build LLM-powered applications easily.'})
print(result)

### Example 2 — Q&A with Persona using `.batch()`

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert {domain} teacher. Give short, clear answers.'),
    ('human',  '{question}'),
])

qa_chain = qa_prompt | llm | StrOutputParser()

# Batch: ask multiple questions in parallel
questions = [
    {'domain': 'Python',       'question': 'What is a decorator?'},
    {'domain': 'Data Science', 'question': 'What is overfitting?'},
]

answers = qa_chain.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q['question']}")
    print(f"A: {a}")
    print('-' * 50)

---
## 6. Best Practices

| Practice | Why |
|---|---|
| Add `StrOutputParser()` at the end | Gets plain string instead of `AIMessage` object |
| Break complex logic into small chains | Easier to debug and reuse |
| Use `RunnableParallel` for independent tasks | Runs concurrently — faster |
| Use `.batch()` for multiple inputs | Much faster than looping `.invoke()` |
| Test sub-chains individually | Debug step by step |

---
## Summary

```
Simple Chain      :  prompt | llm | parser
Sequential Chain  :  chain1 | transform | chain2
Parallel Chain    :  RunnableParallel(a=chain1, b=chain2)
```

> **Golden rule:** Each `|` passes the output of the left as input to the right.
> The whole chain is a Runnable — `.invoke()`, `.batch()`, `.stream()` all work on it.